# AI-Generated Audio Detection System
## Senior Capstone Research Project

This notebook is a thin launcher for the modular Python codebase.
All logic lives in `src/` — this notebook handles Colab-specific setup,
dataset mounting, and training orchestration.

### Quick Start
1. Mount Google Drive
2. Clone/sync the repo
3. Install dependencies
4. Run smoke test to verify pipeline
5. Launch full training

## 1. Environment Setup

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or update the repository
import os

REPO_URL = "https://github.com/foojanbabaeeian/AI-Innovation"
  # UPDATE THIS
REPO_DIR = "/content/AI-Innovation"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

# Install ffmpeg for codec augmentation
!apt-get install -qq ffmpeg

# Verify
!ffmpeg -version | head -1
!python -c "import torch; print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')"

## 2. Dataset Setup

**Option A:** Datasets stored on Google Drive  
**Option B:** Download datasets directly to Colab runtime

In [ ]:
# Option A: Symlink datasets from Google Drive
DRIVE_DATA = "/content/drive/MyDrive/AudioDetection/data"

if os.path.exists(DRIVE_DATA):
    !ln -sf {DRIVE_DATA} data
    print("Linked datasets from Google Drive.")
else:
    print(f"No data found at {DRIVE_DATA}. Using local data/ directory.")
    os.makedirs("data", exist_ok=True)

In [ ]:
# Check which datasets are available
from src.data.download import verify_datasets, print_manual_download_instructions

status = verify_datasets()

# Uncomment to see download instructions for missing datasets:
# print_manual_download_instructions()

## 3. Smoke Test

Run the smoke test to verify the entire pipeline works before launching full training.

In [ ]:
# Run unit tests (uses synthetic data, no real datasets needed)
!python -m pytest tests/test_data_pipeline.py -v --tb=short 2>&1 | tail -30

In [ ]:
# Quick smoke test: load config + verify preprocessing on a single sample
import torch
from src.config import load_config
from src.data.preprocessing import AudioPreprocessor
from src.data.augmentation import CodecAugmentor

config = load_config(overrides={"smoke_test": {"enabled": True}})
preprocessor = AudioPreprocessor.from_config(config)
augmentor = CodecAugmentor.from_config(config)

# Create a synthetic test waveform
test_waveform = torch.randn(1, 64000)
mel = preprocessor.compute_mel_spectrogram(test_waveform)
aug_waveform = augmentor(test_waveform)

print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"Mel spectrogram shape: {mel.shape}")
print(f"Augmented waveform shape: {aug_waveform.shape}")
print(f"FFmpeg available: {augmentor.has_ffmpeg}")
print("\n✓ Pipeline smoke test passed!")

## 4. Training (Have not worked on it yet)

Model architecture and training loop will be added in the next phase.
The training code will use the data pipeline verified above.

In [ ]:
# Placeholder: full training will be launched here
# from src.train import train
# config = load_config()
# train(config)

## 5. Evaluation (Have not worked on it yet)

In [ ]:
# Placeholder: evaluation on test sets
# from src.evaluation.metrics import compute_all_metrics
# results = compute_all_metrics(labels, scores)
# print(f"EER: {results['eer']:.4f}")
# print(f"min-DCF: {results['min_dcf']:.4f}")